# **Data Retrieval**

In [1]:
import os
import pickle
import time

import streamlit as st
import langchain

# LLM
from langchain_openai import OpenAI
from langchain.chat_models import init_chat_model

# Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Document loader
from langchain_community.document_loaders import UnstructuredURLLoader

# Embeddings
from langchain_openai import OpenAIEmbeddings

# Vector store
from langchain_community.vectorstores import FAISS

W0908 15:15:39.357000 56520 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0908 15:15:39.555000 56520 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


C:\Users\meisa\AppData\Local\Temp\ipykernel_56520\1834688414.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader


In [2]:
#load openAI api key
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

In [3]:
# Initialise LLM with required params
llm = OpenAI(temperature=0.9, max_tokens=500, api_key=api_key, base_url=base_url) 
ollama = init_chat_model("llama3.1:8b ", model_provider="ollama", temperature=0)


### (1) Load data

In [4]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load() 
len(data)

2

### (2) Split data to create chunks

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# As data is of type documents we can directly use split_documents over split_text in order to get the chunks.
docs = text_splitter.split_documents(data)

In [6]:
len(docs)

14

In [7]:
docs[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nLoan Offers\n\nMy Alerts\n\nGo Ad-Free\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nLoan against MFs\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\n>->\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol AD Lite\n\nMoneycontrol AD Lite\n\nMoneycontrol PRO\n\nMoneycontrol Super PRO\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpinion\n\nEV Special\n\nEco Pulse\n\nTrending Topics\n\nSensex Today\n\nGE Vernova T&D India Shares\n\nHAL S

### (3) Create embeddings for these chunks and save them to FAISS index

In [9]:
# Create the embeddings of the chunks using openAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=api_key, openai_api_base=base_url)
#embeddings = OpenAIEmbeddings()

# Pass the documents and embeddings inorder to create FAISS vector index
vectorindex_openai = FAISS.from_documents(docs, embeddings)

In [14]:
# ✅ CORRECT: Use FAISS's built-in save method
file_path = "faiss_index"  # This will be a directory

# Save the vector store
vectorindex_openai.save_local(file_path)
print(f"✅ Vector store saved to: {file_path}")

# Load it back later
from langchain_community.vectorstores import FAISS

loaded_vectorstore = FAISS.load_local(
    file_path,
    embeddings=embeddings,  # You need the same embeddings instance
    allow_dangerous_deserialization=True  # Required for FAISS
)
print(f"✅ Vector store loaded from: {file_path}")

✅ Vector store saved to: faiss_index
✅ Vector store loaded from: faiss_index


In [17]:
# Test the vector store immediately after creation
query = "what is the price of Tiago iCNG?"
results = vectorindex_openai.similarity_search(query, k=1)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")
    print("-" * 40)

Query: what is the price of Tiago iCNG?

Result 1:
Content: Trending Topics

Sensex Live

Gift Nifty

Deepa Jewellers IPO Listing

Gold and Silver Prices Today

Karamtara Engineering IPO

Tata Motors launches Punch iCNG, price starts at Rs 7.1 lakh

The Punch ...
Metadata: {'source': 'https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html'}
----------------------------------------


### (4) Retrieve similar embeddings for a given question and call LLM to retrieve final answer

In [27]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

In [30]:
# Turn your FAISS vector store into a retriever
retriever = vectorindex_openai.as_retriever()


# Prompt used to answer based on retrieved context
prompt = ChatPromptTemplate.from_template(
    """
Answer the question using only the provided context.

Context:
{context}

Question:
{input}

Answer:
"""
)


# Combine retrieved documents into the LLM prompt
document_chain = create_stuff_documents_chain(
    llm=ollama,
    prompt=prompt
)


# Create the full retrieval pipeline
chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [31]:
result = chain.invoke({
    "input": "what is the price of Tiago iCNG?"
})

print(result["answer"])

There is no information about the price of the Tiago iCNG in the provided context. The article mentions the launch of the Tata Motors Punch iCNG, but not the Tiago iCNG.


### 5) GUI for the RAG agent

In [ ]:
import os
import time

import streamlit as st

from dotenv import load_dotenv

# Modern LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate

# Retrieval-chain utilities are currently in langchain_classic
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)


# -------------------------------------------------------
# 1. Load environment variables
# -------------------------------------------------------

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

# -------------------------------------------------------
# 2. Streamlit UI
# -------------------------------------------------------

st.title("RockyBot: News Research Tool 📈")

st.sidebar.title("News Article URLs")


# Collect URLs from the sidebar
urls = []

for i in range(3):
    url = st.sidebar.text_input(f"URL {i + 1}")

    if url:
        urls.append(url)


# Button to process URLs
process_url_clicked = st.sidebar.button("Process URLs")


# Folder where FAISS will be stored
faiss_path = "faiss_store_openai"


# Placeholder for status messages
main_placeholder = st.empty()


# -------------------------------------------------------
# 3. Initialize LLM
# -------------------------------------------------------

llm = init_chat_model("qwen3.8:latest", model_provider="ollama", temperature=0)



# -------------------------------------------------------
# 4. Initialize embedding model
# -------------------------------------------------------

embeddings = OpenAIEmbeddings(model="text-embedding-3-large", 
                              api_key=api_key, 
                              openai_api_base=base_url)



# -------------------------------------------------------
# 5. Process URLs
# -------------------------------------------------------

if process_url_clicked:

    if not urls:
        st.warning("Please enter at least one URL.")

    else:

        # -----------------------------------------------
        # Load web pages
        # -----------------------------------------------

        main_placeholder.text("Loading articles... ✅")

        loader = UnstructuredURLLoader(
            urls=urls
        )

        data = loader.load()

        st.write(
            f"Loaded {len(data)} documents."
        )


        # -----------------------------------------------
        # Split documents into chunks
        # -----------------------------------------------

        main_placeholder.text("Splitting text... ✅")

        text_splitter = RecursiveCharacterTextSplitter(

            # Try paragraph, newline, sentence, comma,
            # space, then character-level splitting
            separators=[
                "\n\n",
                "\n",
                ".",
                ",",
                " ",
                ""
            ],

            chunk_size=1000,

            # Some overlap helps preserve context
            chunk_overlap=100,

            length_function=len,
        )


        docs = text_splitter.split_documents(data)

        st.write(
            f"Created {len(docs)} chunks."
        )


        # -----------------------------------------------
        # Create vector embeddings + FAISS index
        # -----------------------------------------------

        main_placeholder.text(
            "Creating embeddings and FAISS index... ✅"
        )

        vectorstore = FAISS.from_documents(
            documents=docs,
            embedding=embeddings
        )


        # -----------------------------------------------
        # Save FAISS index locally
        # -----------------------------------------------

        vectorstore.save_local(
            faiss_path
        )

        main_placeholder.text(
            "FAISS index saved successfully ✅"
        )

        time.sleep(1)


# -------------------------------------------------------
# 6. Ask questions
# -------------------------------------------------------

query = st.text_input(
    "Question:"
)


if query:

    # Make sure FAISS index exists
    if os.path.exists(faiss_path):

        # -----------------------------------------------
        # Load saved FAISS database
        # -----------------------------------------------

        vectorstore = FAISS.load_local(

            faiss_path,

            embeddings,

            # FAISS metadata uses pickle internally.
            # Only enable this for indexes YOU created/trust.
            allow_dangerous_deserialization=True,
        )


        # -----------------------------------------------
        # Convert vector store into a retriever
        # -----------------------------------------------

        retriever = vectorstore.as_retriever(
            search_kwargs={
                "k": 4
            }
        )


        # -----------------------------------------------
        # Prompt for RAG
        # -----------------------------------------------

        prompt = ChatPromptTemplate.from_template(
            """
            You are a news research assistant.

            Answer the user's question using only the
            information contained in the retrieved context.

            If the answer cannot be found in the context,
            say that you do not have enough information.

            Context:
            {context}

            Question:
            {input}

            Answer:
            """
        )


        # -----------------------------------------------
        # Chain that sends retrieved documents to LLM
        # -----------------------------------------------

        document_chain = create_stuff_documents_chain(
            llm=llm,
            prompt=prompt,
        )


        # -----------------------------------------------
        # Full RAG chain:
        #
        # Question
        #   ↓
        # Retriever
        #   ↓
        # Relevant documents
        #   ↓
        # Prompt
        #   ↓
        # LLM
        #   ↓
        # Answer
        # -----------------------------------------------

        retrieval_chain = create_retrieval_chain(
            retriever,
            document_chain
        )


        # -----------------------------------------------
        # Execute RAG pipeline
        # -----------------------------------------------

        result = retrieval_chain.invoke(
            {
                "input": query
            }
        )


        # -----------------------------------------------
        # Display answer
        # -----------------------------------------------

        st.header("Answer")

        st.write(
            result["answer"]
        )


        # -----------------------------------------------
        # Display sources
        # -----------------------------------------------

        source_docs = result.get(
            "context",
            []
        )


        if source_docs:

            st.subheader("Sources:")

            # Avoid displaying duplicate URLs
            sources = []

            for doc in source_docs:

                source = doc.metadata.get(
                    "source"
                )

                if source and source not in sources:
                    sources.append(source)


            for source in sources:
                st.write(source)

    else:

        st.warning(
            "Please process the URLs first."
        )